In [ ]:
import numpy as np
import pandas as pd
import random
from joblib import Parallel, delayed
from regression_common import ridge_pair, lasso_coefficients, sparse_indices, discrepancy_candidate


In [ ]:

np.random.seed(24)
n = 1000
p = 1000


cov_matrix = np.full((p, p), 0.5)
np.fill_diagonal(cov_matrix, 2.0)

mean_vector                = np.zeros(p)
X                          = np.random.multivariate_normal(mean_vector, cov_matrix, n)

U, S, Vt = np.linalg.svd(X)


condition_number = S[0] / S[-1]
print("Condition number:", condition_number)


In [ ]:

np.random.seed(25)
beta = np.random.uniform(low=-2, high=2, size=p)
print(np.linalg.norm(beta))


In [ ]:

np.random.seed(26)
mean    = 0
std_dev = 2


e = np.random.normal(mean, std_dev, n)
norm_e = np.linalg.norm(e)
Y = np.dot(X, beta) + e
print(norm_e)


In [ ]:
s1, s2 = X.shape


In [ ]:

def least_square(X, Y):
    beta = np.linalg.lstsq(X, Y, rcond=None)[0]
    return beta
beta_LS = least_square(X, Y)
norm_LS = np.linalg.norm(beta_LS - beta)
print(norm_LS)
sigma_LS = np.linalg.norm(Y - np.dot(X, beta_LS))
sigma_LS = sigma_LS / np.sqrt(X.shape[0])
print(sigma_LS)


In [ ]:

U, S, Vt = np.linalg.svd(X)


def compute_norm_cut(k):

    S_truncated = np.zeros_like(S)
    S_truncated[:k] = S[:k]


    Sigma_truncated = np.zeros((U.shape[0], Vt.shape[0]))
    np.fill_diagonal(Sigma_truncated, S_truncated)


    S_cut = Sigma_truncated.T @ Sigma_truncated
    S_inv = np.zeros_like(S_cut)
    S_inv[S_cut != 0] = 1 / S_cut[S_cut != 0]


    beta_cut = Vt.T @ S_inv @ Vt @ X.T @ Y


    norm_cut = np.linalg.norm(beta_cut - beta)

    return norm_cut, k


results = Parallel(n_jobs=-1)(delayed(compute_norm_cut)(k) for k in range(1000, 0, -1))


min_norm_cut, best_k = min(results, key=lambda x: x[0])


print(f"Minimum norm_cut: {min_norm_cut}, Selected k: {best_k}")


In [ ]:
# Spectral cut-off regression


k = best_k


S_truncated = np.zeros_like(S)
S_truncated[:k] = S[:k]


Sigma = np.zeros((U.shape[0], Vt.shape[0]))
Sigma_truncated = np.zeros((U.shape[0], Vt.shape[0]))
np.fill_diagonal(Sigma, S)
np.fill_diagonal(Sigma_truncated, S_truncated)
X_truncated = U @ Sigma_truncated @ Vt
S_cut = Sigma_truncated.T@Sigma_truncated
S_inv = np.zeros_like(S_cut)
S_inv[S_cut != 0] = 1 / S_cut[S_cut != 0]
beta_cut = Vt.T @ S_inv @ Vt @ X.T @ Y
norm_cut = np.linalg.norm(beta_cut - beta)
I = np.eye(s2)
# At a fixed SC cutoff, r_alpha**2 = r_alpha, so G_alpha = g_alpha.
beta_cut_debias = beta_cut.copy()
norm_cut_debias = np.linalg.norm(beta_cut_debias - beta)
print(norm_cut, norm_cut_debias)
sigma_cut = np.linalg.norm(Y - X @ beta_cut)
sigma_cut = sigma_cut / np.sqrt(X.shape[0])
print(sigma_cut)
sigma_cut_debias = np.linalg.norm(Y - X @ beta_cut_debias)
sigma_cut_debias = sigma_cut_debias / np.sqrt(X.shape[0])
print(sigma_cut_debias)


In [ ]:
# Ridge Regression
C = 50000
m = 10
def compute_mse(c, X, Y):
    beta_r, _ = ridge_pair(X, Y, c / X.shape[0])
    return np.linalg.norm(beta_r - beta)


mse_list = Parallel(n_jobs=-1)(delayed(compute_mse)(c, X, Y) for c in range(0, C + 1, m))

best_c = np.argmin(mse_list)*m
print("The value of C that minimizes the mean squared error is:", best_c)


In [ ]:

def ridge(X, Y, C):
    from regression_common import ridge_pair
    return ridge_pair(X, Y, C / X.shape[0])

ridge_alpha = best_c / X.shape[0]
beta_Ridge, beta_Ridge_debias = ridge(X, Y, best_c)
norm_Ridge = np.linalg.norm(beta_Ridge - beta)
norm_Ridge_debias = np.linalg.norm(beta_Ridge_debias - beta)
print(norm_Ridge, norm_Ridge_debias)
sigma_Ridge = np.linalg.norm(Y - np.dot(X, beta_Ridge))
sigma_Ridge = sigma_Ridge / np.sqrt(X.shape[0])
print(sigma_Ridge)
sigma_Ridge_debias = np.linalg.norm(Y - np.dot(X, beta_Ridge_debias))
sigma_Ridge_debias = sigma_Ridge_debias / np.sqrt(X.shape[0])
print(sigma_Ridge_debias)


In [ ]:
# Lasso Regression
from sklearn.linear_model import Lasso

C = 1000
m = 1
def compute_mse(c, X, Y):
    beta_lasso = lasso_coefficients(X, Y, c / X.shape[0])
    return np.linalg.norm(beta_lasso - beta)


mse_list = Parallel(n_jobs=-1)(delayed(compute_mse)(c, X, Y) for c in range(0, C + 1, m))

best_c = np.argmin(mse_list)*m
print("The value of C is:", best_c)


In [ ]:
lasso_alpha = best_c / X.shape[0]
beta_lasso = lasso_coefficients(X, Y, lasso_alpha)
norm_Lasso = np.linalg.norm(beta_lasso - beta)
print(norm_Lasso)
sigma_Lasso = np.linalg.norm(Y - np.dot(X, beta_lasso))
sigma_Lasso = sigma_Lasso / np.sqrt(X.shape[0])
print(sigma_Lasso)


In [ ]:

t        = 3e-6
max_iter = 100000

def landweber(X, Y, t, max_iter):
    s = X.shape
    m = s[1]
    beta_cur = np.zeros(m)
    completed_steps = 0
    target_steps = max_iter
    turning_found = False
    for k in range(1, max_iter + 1):
        beta_prev = beta_cur
        beta_cur = beta_prev + t * np.dot(X.T, (Y - np.dot(X, beta_prev)))
        completed_steps = k
        if not turning_found and completed_steps >= 2:
            if np.linalg.norm(beta_cur - beta) >= np.linalg.norm(beta_prev - beta):
                target_steps = min(max_iter, max(completed_steps - 1, 500))
                turning_found = True
                if target_steps < completed_steps:
                    return beta_prev, completed_steps - 1
        if completed_steps >= target_steps:
            break
    return beta_cur, completed_steps
def landweber_debias(X, Y, t, max_iter):
    s = X.shape
    m = s[1]
    beta_cur = np.zeros(m)
    for k in range(1, max_iter+1):
        beta_prev = beta_cur
        beta_cur = beta_prev + t * np.dot(X.T, (Y - np.dot(X, beta_prev)))
    return beta_cur
beta_Landweber, k_Landweber = landweber(X, Y, t, max_iter)
beta_Landweber_debias = landweber_debias(X, Y, t, 2*k_Landweber)
norm_Landweber    = np.linalg.norm(beta_Landweber - beta)
norm_Landweber_debias  = np.linalg.norm(beta_Landweber_debias - beta)

print(norm_Landweber, k_Landweber, norm_Landweber_debias)
sigma_Landeweber = np.linalg.norm(Y - np.dot(X, beta_Landweber))
sigma_Landeweber = sigma_Landeweber / np.sqrt(X.shape[0])
print(sigma_Landeweber)
sigma_Landeweber_debias = np.linalg.norm(Y - np.dot(X, beta_Landweber_debias))
sigma_Landeweber_debias = sigma_Landeweber_debias / np.sqrt(X.shape[0])
print(sigma_Landeweber_debias)


In [ ]:

t        = 3e-6
max_iter = 100000

def showalter(X, Y, t, max_iter):
    s = X.shape
    m = s[1]
    beta_cur = np.zeros(m)
    Z1 = np.dot(X.T, Y)
    Z2 = np.dot(X.T, X)
    completed_steps = 0
    target_steps = max_iter
    turning_found = False
    for k in range(1, max_iter + 1):
        beta_prev = beta_cur
        K1 = Z1 - np.dot(Z2, beta_prev)
        K2 = Z1 - np.dot(Z2, (beta_prev + 0.5 * t * K1))
        K3 = Z1 - np.dot(Z2, (beta_prev + 0.5 * t * K2))
        K4 = Z1 - np.dot(Z2, (beta_prev + t * K3))
        beta_cur = beta_prev + t / 6 * (K1 + 2 * K2 + 2 * K3 + K4)
        completed_steps = k
        if not turning_found and completed_steps >= 2:
            if np.linalg.norm(beta_cur - beta) >= np.linalg.norm(beta_prev - beta):
                target_steps = min(max_iter, max(completed_steps - 1, 500))
                turning_found = True
                if target_steps < completed_steps:
                    return beta_prev, completed_steps - 1
        if completed_steps >= target_steps:
            break
    return beta_cur, completed_steps
def showalter_debias(X, Y, t, max_iter):
    s = X.shape
    m = s[1]
    beta_cur = np.zeros(m)
    Z1 = np.dot(X.T, Y)
    Z2 = np.dot(X.T, X)
    for k in range(1, max_iter+1):
        beta_prev = beta_cur
        K1 = Z1 - np.dot(Z2, beta_prev)
        K2 = Z1 - np.dot(Z2, (beta_prev + 0.5 * t * K1))
        K3 = Z1 - np.dot(Z2, (beta_prev + 0.5 * t * K2))
        K4 = Z1 - np.dot(Z2, (beta_prev + t * K3))
        beta_cur = beta_prev + t / 6 * (K1 + 2 * K2 + 2 * K3 + K4)
    return beta_cur
beta_showalter, k_showalter = showalter(X, Y, t, max_iter)
beta_showalter_debias = showalter_debias(X, X@beta_showalter, t, k_showalter)
beta_showalter_debias = 2*beta_showalter-beta_showalter_debias
norm_showalter = np.linalg.norm(beta_showalter - beta)
norm_showalter_debias = np.linalg.norm(beta_showalter_debias - beta)
print(norm_showalter, k_showalter, norm_showalter_debias)
sigma_showalter = np.linalg.norm(Y - np.dot(X, beta_showalter))
sigma_showalter = sigma_showalter / np.sqrt(X.shape[0])
print(sigma_showalter)
sigma_showalter_debias = np.linalg.norm(Y - np.dot(X, beta_showalter_debias))
sigma_showalter_debias = sigma_showalter_debias / np.sqrt(X.shape[0])
print(sigma_showalter_debias)


In [ ]:

t        = 5e-4
max_iter = 100000

def hbf(X, Y, t, max_iter):
    s = X.shape
    m = s[1]
    eta = 5
    beta_cur = np.zeros(m)
    A11 = np.zeros((m, m))
    A12 = np.eye(m)
    A21 = -np.dot(X.T, X)
    A22 = -eta * np.eye(m)
    A = np.vstack((np.hstack((A11, A12)), np.hstack((A21, A22))))
    b = np.hstack((np.zeros(m), np.dot(X.T, Y)))
    z = np.zeros(2 * m)

    completed_steps = 0
    target_steps = max_iter
    turning_found = False
    for k in range(1, max_iter + 1):
        beta_prev = beta_cur
        K1 = np.dot(A, z) + b
        K2 = np.dot(A, (z + 0.5 * t * K1)) + b
        K3 = np.dot(A, (z + 0.5 * t * K2)) + b
        K4 = np.dot(A, (z + t * K3)) + b
        z = z + t / 6 * (K1 + 2 * K2 + 2 * K3 + K4)
        beta_cur = z[:m]
        completed_steps = k
        if not turning_found and completed_steps >= 2:
            if np.linalg.norm(beta_cur - beta) >= np.linalg.norm(beta_prev - beta):
                target_steps = min(max_iter, max(completed_steps - 1, 500))
                turning_found = True
                if target_steps < completed_steps:
                    return beta_prev, completed_steps - 1
        if completed_steps >= target_steps:
            break
    return beta_cur, completed_steps

def hbf_debias(X, Y, t, max_iter):
    s = X.shape
    m = s[1]
    eta = 5

    A11 = np.zeros((m, m))
    A12 = np.eye(m)
    A21 = -np.dot(X.T, X)
    A22 = -eta * np.eye(m)
    A = np.vstack((np.hstack((A11, A12)), np.hstack((A21, A22))))
    b = np.hstack((np.zeros(m), np.dot(X.T, Y)))
    z = np.zeros(2 * m)

    beta_cur = np.zeros(X.shape[1])
    for k in range(1, max_iter+1):
        K1 = np.dot(A, z) + b
        K2 = np.dot(A, (z + 0.5 * t * K1)) + b
        K3 = np.dot(A, (z + 0.5 * t * K2)) + b
        K4 = np.dot(A, (z + t * K3)) + b
        z = z + t / 6 * (K1 + 2 * K2 + 2 * K3 + K4)
        beta_cur = z[:m]
    return beta_cur
beta_hbf, k_hbf = hbf(X, Y, t, max_iter)
beta_hbf_debias = hbf_debias(X, X@beta_hbf, t, k_hbf)
beta_hbf_debias = 2*beta_hbf - beta_hbf_debias
norm_hbf = np.linalg.norm(beta_hbf - beta)
norm_hbf_debias = np.linalg.norm(beta_hbf_debias - beta)
print(norm_hbf, k_hbf, norm_hbf_debias)
sigma_hbf = np.linalg.norm(Y - np.dot(X, beta_hbf))
sigma_hbf = sigma_hbf / np.sqrt(X.shape[0])
print(sigma_hbf)
sigma_hbf_debias = np.linalg.norm(Y - np.dot(X, beta_hbf_debias))
sigma_hbf_debias = sigma_hbf_debias / np.sqrt(X.shape[0])
print(sigma_hbf_debias)


In [ ]:
# Acceleration regression of order kappa
t        = 5e-5
max_iter = 2000
kappa    = 0.5
I        = np.eye(s2)
def D(k):
    if k == 0:
        return np.zeros((s2, s2))
    else:
        D = np.linalg.inv((t * (k * t) ** kappa) * X.T @ X + I + (((k * t) ** (-kappa)) - kappa) / k * I)
        return D


D_array = Parallel(n_jobs=-1)(delayed(D)(k) for k in range(2001))


D = np.array(D_array)


In [ ]:
def ARk(X, Y, t, k_max):
    s = X.shape
    m = s[1]
    beta_prev = np.zeros(m)
    I = np.eye(m)
    v_prev = np.zeros(m)
    beta_cur = np.zeros(m)
    completed_steps = 0
    target_steps = k_max
    turning_found = False
    for k in range(1, k_max + 1):
        beta_prev = beta_cur
        beta_cur = beta_prev + k * t * v_prev
        v_cur = D[k] @ v_prev + t * D[k] @ np.dot(X.T, (Y - np.dot(X, beta_prev)))
        v_prev = v_cur
        completed_steps = k
        if not turning_found and completed_steps >= 2:
            if np.linalg.norm(beta_cur - beta) >= np.linalg.norm(beta_prev - beta):
                target_steps = min(k_max, max(completed_steps - 1, 500))
                turning_found = True
                if target_steps < completed_steps:
                    return beta_prev, completed_steps - 1
        if completed_steps >= target_steps:
            break
    return beta_cur, completed_steps

def ARk_debias(X, Y, t, k_max):
    s = X.shape
    m = s[1]
    beta_prev = np.zeros(m)
    I = np.eye(m)
    v_prev = np.zeros(m)
    beta_cur = np.zeros(m)
    for k in range(1, k_max+1):
        beta_prev = beta_cur
        beta_cur = beta_prev + k * t * v_prev
        v_cur = D[k] @ v_prev + t * D[k] @ np.dot(X.T, (Y - np.dot(X, beta_prev)))
        v_prev = v_cur
    return beta_cur

beta_ARk, k_ARk = ARk(X, Y, t, max_iter)
beta_ARk_debias = ARk_debias(X, X @ beta_ARk, t, k_ARk)
beta_ARk_debias = 2 * beta_ARk - beta_ARk_debias
norm_ARk = np.linalg.norm(beta_ARk - beta)
norm_ARk_debias = np.linalg.norm(beta_ARk_debias - beta)
print(norm_ARk, k_ARk, norm_ARk_debias)
sigma_ARk = np.linalg.norm(Y - np.dot(X, beta_ARk))
sigma_ARk = sigma_ARk / np.sqrt(X.shape[0])
print(sigma_ARk)
sigma_ARk_debias = np.linalg.norm(Y - np.dot(X, beta_ARk_debias))
sigma_ARk_debias = sigma_ARk_debias / np.sqrt(X.shape[0])
print(sigma_ARk_debias)


In [ ]:

t        = 5e-4
max_iter = 100000

def second(X, Y, t, k_max):
    r = X.shape
    m = r[1]
    s = 1.5
    beta_cur = np.zeros(m)
    v_cur    = np.zeros(m)
    q_cur    = np.zeros(m)
    completed_steps = 0
    target_steps = k_max
    turning_found = False
    for k in range(1, k_max + 1):
        beta_prev = beta_cur
        p        =  2*k/(2*k+2*s+1)
        q_half   =  p*q_cur + p*(t / 2) * np.dot(X.T, (Y - np.dot(X, beta_cur)))
        beta_cur = beta_prev + t * q_half
        a        = (2*k-2*s+1)*(k+1)/(2*k+2*s+3)/k
        v_cur   =  beta_cur + 2*t*a*q_half
        q_cur    =  (2*k+1-2*s)/(2*k+2)*q_half +(t / 2) * np.dot(X.T, (Y - np.dot(X, v_cur)))
        completed_steps = k
        if not turning_found and completed_steps >= 2:
            if np.linalg.norm(beta_cur - beta) >= np.linalg.norm(beta_prev - beta):
                target_steps = min(k_max, max(completed_steps - 1, 500))
                turning_found = True
                if target_steps < completed_steps:
                    return beta_prev, completed_steps - 1
        if completed_steps >= target_steps:
            break
    return beta_cur, completed_steps

def second_debias(X, Y, t, k_max):
    r = X.shape
    m = r[1]
    s = 1.5
    beta_cur = np.zeros(m)
    v_cur    = np.zeros(m)
    q_cur    = np.zeros(m)

    for k in range(1, k_max+1):
        p        =  2*k/(2*k+2*s+1)
        q_half   =  p*q_cur + p*(t / 2) * np.dot(X.T, (Y - np.dot(X, beta_cur)))
        beta_cur = beta_cur + t * q_half
        a        = (2*k-2*s+1)*(k+1)/(2*k+2*s+3)/k
        v_cur   =  beta_cur + 2*t*a*q_half
        q_cur    =  (2*k+1-2*s)/(2*k+2)*q_half +(t / 2) * np.dot(X.T, (Y - np.dot(X, v_cur)))
    return beta_cur

beta_second, k_second = second(X, Y, t, max_iter)
beta_second_debias = second_debias(X, X@beta_second, t, k_second)
beta_second_debias = 2*beta_second - beta_second_debias
norm_second = np.linalg.norm(beta_second - beta)
norm_second_debias = np.linalg.norm(beta_second_debias - beta)
print(norm_second, k_second, norm_second_debias)
sigma_second = np.linalg.norm(Y - np.dot(X, beta_second))
sigma_second = sigma_second / np.sqrt(X.shape[0])
print(sigma_second)
sigma_second_debias = np.linalg.norm(Y - np.dot(X, beta_second_debias))
sigma_second_debias = sigma_second_debias / np.sqrt(X.shape[0])
print(sigma_second_debias)


In [ ]:

nominal_nesterov_step        = 3e-6
t = min(nominal_nesterov_step, 0.99 / S[0]**2)
effective_nesterov_step = t
max_iter = 100000
omega    = 3       # omega

def Nesterov(X, Y, t, k_max, beta_reference, omega, k_min=500):
    beta_prev = np.zeros(X.shape[1])
    beta_cur = np.zeros(X.shape[1])
    completed_steps = 0
    target_steps = k_max
    turning_found = False
    for k in range(1, k_max + 1):
        # k counts gradient updates; the first gives beta_1 = t * X.T @ Y.
        momentum = 0.0 if k == 1 else (k - 2) / (k - 1 + omega)
        z_k = beta_cur + momentum * (beta_cur - beta_prev)
        beta_next = z_k + t * (X.T @ (Y - X @ z_k))
        beta_prev, beta_cur = beta_cur, beta_next
        completed_steps = k
        if not turning_found and completed_steps >= 2:
            if np.linalg.norm(beta_cur - beta_reference) >= np.linalg.norm(beta_prev - beta_reference):
                target_steps = min(k_max, max(completed_steps - 1, k_min))
                turning_found = True
                if target_steps < completed_steps:
                    return beta_prev, completed_steps - 1
        if completed_steps >= target_steps:
            break
    return beta_cur, completed_steps

def Nesterov_debias(X, Y, t, k_max, omega):
    beta_prev = np.zeros(X.shape[1])
    beta_cur = np.zeros(X.shape[1])
    for k in range(1, k_max + 1):
        momentum = 0.0 if k == 1 else (k - 2) / (k - 1 + omega)
        z_k = beta_cur + momentum * (beta_cur - beta_prev)
        beta_next = z_k + t * (X.T @ (Y - X @ z_k))
        beta_prev, beta_cur = beta_cur, beta_next
    return beta_cur

beta_Nesterov, k_Nesterov = Nesterov(X, Y, t, max_iter, beta, omega)
beta_Nesterov_debias = Nesterov_debias(X, X@beta_Nesterov, t, k_Nesterov, omega)
beta_Nesterov_debias = 2*beta_Nesterov - beta_Nesterov_debias
norm_Nesterov = np.linalg.norm(beta_Nesterov - beta)
norm_Nesterov_debias = np.linalg.norm(beta_Nesterov_debias - beta)
print(norm_Nesterov, k_Nesterov, norm_Nesterov_debias)
sigma_Nesterov = np.linalg.norm(Y - np.dot(X, beta_Nesterov))
sigma_Nesterov = sigma_Nesterov / np.sqrt(X.shape[0])
print(sigma_Nesterov)
sigma_Nesterov_debias = np.linalg.norm(Y - np.dot(X, beta_Nesterov_debias))
sigma_Nesterov_debias = sigma_Nesterov_debias / np.sqrt(X.shape[0])
print(sigma_Nesterov_debias)


In [ ]:
from scipy.special import gamma


t        = 5e-4
max_iter = 2000
theta    = 1.8      # theta
A        = np.zeros((max_iter+1, max_iter+1))
B        = np.zeros((max_iter, max_iter+1))
g        = 1/gamma(theta)
k1       = t**(theta)/(theta*(theta+1))
k2       = t**(theta)/theta

for k in range(max_iter):
    for j in range(k+2):
        if j == 0:
            d = k**(theta+1)-(k-theta)*(k+1)**(theta)
            A[j, k+1] = k1*d
        elif j < k+1:
            d = (k-j+2)**(theta+1)+(k-j)**(theta+1)-2*(k-j+1)**(theta+1)
            A[j, k+1] = k1*d
        elif j == k+1:
            A[j, k+1] = k1
    for i in range(k+1):
        if i < k+1:
            d = (k-i+1)**theta-(k-i)**theta
            B[i, k+1] = k2*d


def Frac(X, Y, k_max):
    s  = X.shape
    m  = s[1]
    b  = np.zeros((m, k_max+1))
    bp = np.zeros((m, k_max+1))
    beta_cur = np.zeros(m)
    completed_steps = 0
    target_steps = k_max
    turning_found = False
    for k in range(k_max):
        beta_prev = beta_cur
        bp[:, k+1] = g*sum([B[j, k+1]*np.dot(X.T, (Y - np.dot(X, b[:,j]))) for j in range(k+1)])
        b[:, k+1]  = g*(A[k+1, k+1]*np.dot(X.T, (Y - np.dot(X, bp[:,k+1])))+sum([A[j, k+1]*np.dot(X.T, (Y - np.dot(X, b[:,j]))) for j in range(k+1)]))
        beta_cur   = b[:, k+1]
        completed_steps = k + 1
        completed_steps = k + 1
        if not turning_found and completed_steps >= 2:
            if np.linalg.norm(beta_cur - beta) >= np.linalg.norm(beta_prev - beta):
                target_steps = min(k_max, max(completed_steps - 1, 500))
                turning_found = True
                if target_steps < completed_steps:
                    return beta_prev, completed_steps - 1
        if completed_steps >= target_steps:
            break
    return beta_cur, completed_steps
def Frac_debias(X, Y, k_max):
    s  = X.shape
    m  = s[1]
    b  = np.zeros((m, k_max+1))
    bp = np.zeros((m, k_max+1))
    beta_cur = np.zeros(m)
    for k in range(k_max):
        bp[:, k+1] = g*sum([B[j, k+1]*np.dot(X.T, (Y - np.dot(X, b[:,j]))) for j in range(k+1)])
        b[:, k+1]  = g*(A[k+1, k+1]*np.dot(X.T, (Y - np.dot(X, bp[:,k+1])))+sum([A[j, k+1]*np.dot(X.T, (Y - np.dot(X, b[:,j]))) for j in range(k+1)]))
        beta_cur   = b[:, k+1]
    return beta_cur

beta_Frac, k_Frac = Frac(X, Y, max_iter)
beta_Frac_debias = Frac_debias(X, X@beta_Frac, k_Frac)
beta_Frac_debias = 2*beta_Frac - beta_Frac_debias
norm_Frac = np.linalg.norm(beta_Frac - beta)
norm_Frac_debias = np.linalg.norm(beta_Frac_debias - beta)
print(norm_Frac, k_Frac, norm_Frac_debias)
sigma_Frac = np.linalg.norm(Y - np.dot(X, beta_Frac))
sigma_Frac = sigma_Frac / np.sqrt(X.shape[0])
print(sigma_Frac)
sigma_Frac_debias = np.linalg.norm(Y - np.dot(X, beta_Frac_debias))
sigma_Frac_debias = sigma_Frac_debias / np.sqrt(X.shape[0])
print(sigma_Frac_debias)



In [ ]:
print(f"LS                   & {norm_LS:.4f} &  ~    &  ~            \\\\")
print(f"SC                   & {norm_cut:.4f} &  ~    &  {norm_cut_debias:.4f}   \\\\")
print(f"Lasso                & {norm_Lasso:.4f} &  ~    &  ~                \\\\")
print(f"Ridge                & {norm_Ridge:.4f} &  ~    &  {norm_Ridge_debias:.4f}    \\\\")
print(f"Landweber            & {norm_Landweber:.4f}   &  {k_Landweber} &  {norm_Landweber_debias:.4f}   \\\\")
print(f"Showalter            & {norm_showalter:.4f}   &  {k_showalter} &  {norm_showalter_debias:.4f}   \\\\")
print(f"HBF                  & {norm_hbf:.4f}   &  {k_hbf}  &  {norm_hbf_debias:.4f}  \\\\")
print(f"$\\textrmAR^\\kappa$ & {norm_ARk:.4f}   &  {k_ARk}  &  {norm_ARk_debias:.4f}   \\\\")
print(f"SOAR                 & {norm_second:.4f}   &  {k_second}  &  {norm_second_debias:.4f}  \\\\")
print(f"Nesterov             & {norm_Nesterov:.4f}   &  {k_Nesterov} &  {norm_Nesterov_debias:.4f}  \\\\")
print(f"FAR                  & {norm_Frac:.4f}   &  {k_Frac}  &  {norm_Frac_debias:.4f}   \\\\   \\hline")
